# Loading Data

In [1]:
# USE THIS WHEN RUNNING ON COLAB

"""
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/data-dump")

print("Path to dataset files:", path)

df = pd.read_csv(path + '/combined_ipl_mbm_bat.csv')"""


'\nimport kagglehub\n\n# Download latest version\npath = kagglehub.dataset_download("s1nghayush/data-dump")\n\nprint("Path to dataset files:", path)\n\ndf = pd.read_csv(path + \'/combined_ipl_mbm_bat.csv\')'

In [2]:
# USE THIS WHEN RUNNING LOCALLY

import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\mbm\combined_ipl_mbm_bat.csv")

In [3]:
df['date'] = pd.to_datetime(df['date'])

df['boundaries'] = df['fours'] + df['sixes']

df.drop(columns = ['fours', 'sixes', 'sr', 'bowler_id', 'bowler'], inplace=True)

players_detail = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\register_data\player_details.csv")
name_corrector = dict(zip(players_detail['key_cricinfo'], players_detail['cricbuzz_name']))
df['batter'] = df['batter_id'].map(name_corrector)

# Career Stats

In [4]:
df['isOut'] = df['isOut'].replace('DNB', 'False')
df['isOut'] = df['isOut'].astype(str).map({'True': 1, 'False': 0}).fillna(0).astype(int)

In [5]:
# Sort data by batter and match date
df.sort_values(by=['batter_id', 'date'], inplace=True)

# Compute cumulative stats for each batter
df['cumulative_runs'] = df.groupby('batter_id')['runs'].cumsum() - df['runs']
df['cumulative_dismissals'] = df.groupby('batter_id')['isOut'].cumsum() - df['isOut']
df['cumulative_matches'] = df.groupby('batter_id').cumcount()
df['cumulative_balls'] = df.groupby('batter_id')['balls'].cumsum() - df['balls']
df['cumulative_boundaries'] = df.groupby('batter_id')['boundaries'].cumsum() - df['boundaries']

# Compute batting statistics
df['career_avg'] = df['cumulative_runs'] / df['cumulative_dismissals'].replace(0, 1)
df['career_rpm'] = df['cumulative_runs'] / df['cumulative_matches']
df['career_sr'] = (df['cumulative_runs'] * 100) / df['cumulative_balls'].replace(0, 1)
df['career_bpm'] = df['cumulative_boundaries'] / df['cumulative_matches']
df['career_bfpm'] = df['cumulative_balls'] / df['cumulative_matches']

# Drop temporary columns
df.drop(columns=['cumulative_runs',	'cumulative_dismissals',	'cumulative_balls',	'cumulative_matches',	'cumulative_boundaries'], inplace=True)

# Columns to process
career_cols = ['career_avg', 'career_bfpm', 'career_sr', 'career_rpm', 'career_bpm']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in career_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['batter']).head(1).index, career_cols] = pd.NA

In [6]:
df[df['batter_id'] == 234675].head()

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,isOut,wicketType,venue,boundaries,career_avg,career_rpm,career_sr,career_bpm,career_bfpm
14317,2008-04-19,335984,Rajasthan Royals,Delhi Capitals,1,234675,Ravindra Jadeja,29,23,1,bowled,Arun Jaitley Stadium,4,NaN,NaN,NaN,NaN,NaN
14394,2008-04-21,335987,Rajasthan Royals,Punjab Kings,2,234675,Ravindra Jadeja,36,25,0,not out,Sawai Mansingh Stadium,6,29.0,29.0,126.086957,4.000000,23.00
14460,2008-04-24,335990,Rajasthan Royals,Deccan Chargers,2,234675,Ravindra Jadeja,1,3,1,bowled,Rajiv Gandhi International Stadium,0,65.0,32.5,135.416667,5.000000,24.00
14506,2008-04-26,335992,Rajasthan Royals,Royal Challengers Bangalore,2,234675,Ravindra Jadeja,0,0,0,DNB,M Chinnaswamy Stadium,0,33.0,22.0,129.411765,3.333333,17.00
14669,2008-05-01,336000,Rajasthan Royals,Kolkata Knight Riders,1,234675,Ravindra Jadeja,33,19,1,bowled,Sawai Mansingh Stadium,4,33.0,16.5,129.411765,2.500000,12.75


In [7]:
df[df['batter_id'] == 234675].info() # NOTE - Career columns can have one row of null values for each batter


<class 'pandas.core.frame.DataFrame'>
Int64Index: 238 entries, 14317 to 14143
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         238 non-null    datetime64[ns]
 1   match_id     238 non-null    int64         
 2   bat_team     238 non-null    object        
 3   bowl_team    238 non-null    object        
 4   inns         238 non-null    int64         
 5   batter_id    238 non-null    int64         
 6   batter       238 non-null    object        
 7   runs         238 non-null    int64         
 8   balls        238 non-null    int64         
 9   isOut        238 non-null    int32         
 10  wicketType   238 non-null    object        
 11  venue        238 non-null    object        
 12  boundaries   238 non-null    int64         
 13  career_avg   237 non-null    float64       
 14  career_rpm   237 non-null    float64       
 15  career_sr    237 non-null    float64       
 16  ca

# Innings Stats

In [8]:
# Sort data by batter and match date
df.sort_values(by=['batter', 'date'], inplace=True)

# Compute cumulative stats for each batter
df['cumulative_runs'] = df.groupby(['batter_id', 'inns'])['runs'].cumsum() - df['runs']
df['cumulative_dismissals'] = df.groupby(['batter_id', 'inns'])['isOut'].cumsum() - df['isOut']
df['cumulative_matches'] = df.groupby(['batter_id', 'inns']).cumcount()
df['cumulative_balls'] = df.groupby(['batter_id', 'inns'])['balls'].cumsum() - df['balls']
df['cumulative_boundaries'] = df.groupby(['batter_id', 'inns'])['boundaries'].cumsum() - df['boundaries']

# Compute batting statistics
df['inns_avg'] = df['cumulative_runs'] / df['cumulative_dismissals'].replace(0, 1)
df['inns_rpm'] = df['cumulative_runs'] / df['cumulative_matches']
df['inns_sr'] = (df['cumulative_runs'] * 100) / df['cumulative_balls'].replace(0, 1)
df['inns_bpm'] = df['cumulative_boundaries'] / df['cumulative_matches']
df['inns_bfpm'] = df['cumulative_balls'] / df['cumulative_matches']

# Drop temporary columns
df.drop(columns=['cumulative_runs',	'cumulative_dismissals',	'cumulative_balls',	'cumulative_matches',	'cumulative_boundaries'], inplace=True)

# Columns to process
inns_cols = ['inns_avg', 'inns_bfpm', 'inns_sr', 'inns_rpm', 'inns_bpm']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in inns_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['batter', 'inns']).head(1).index, inns_cols] = pd.NA

In [9]:
df[(df['batter_id'] == 234675) & (df['inns'] == 1)].head()

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,isOut,...,career_avg,career_rpm,career_sr,career_bpm,career_bfpm,inns_avg,inns_rpm,inns_sr,inns_bpm,inns_bfpm
14317,2008-04-19,335984,Rajasthan Royals,Delhi Capitals,1,234675,Ravindra Jadeja,29,23,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14669,2008-05-01,336000,Rajasthan Royals,Kolkata Knight Riders,1,234675,Ravindra Jadeja,33,19,1,...,33.0,16.500000,129.411765,2.500000,12.750000,29.000000,29.000000,126.086957,4.000000,23.000000
14845,2008-05-07,336008,Rajasthan Royals,Mumbai Indians,1,234675,Ravindra Jadeja,0,1,1,...,33.0,16.500000,141.428571,2.333333,11.666667,31.000000,31.000000,147.619048,4.000000,21.000000
15176,2008-05-17,336023,Rajasthan Royals,Royal Challengers Bangalore,1,234675,Ravindra Jadeja,0,0,0,...,21.0,11.666667,136.363636,1.666667,8.555556,20.666667,20.666667,144.186047,2.666667,14.333333
15484,2008-05-30,336038,Rajasthan Royals,Delhi Capitals,1,234675,Ravindra Jadeja,7,5,1,...,25.6,10.666667,131.958763,1.333333,8.083333,20.666667,15.500000,144.186047,2.000000,10.750000


In [10]:
df[df['batter_id'] == 234675].info() # NOTE - Innings stats can have two rows of null values for each batter, one for each innings

<class 'pandas.core.frame.DataFrame'>
Int64Index: 238 entries, 14317 to 14143
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         238 non-null    datetime64[ns]
 1   match_id     238 non-null    int64         
 2   bat_team     238 non-null    object        
 3   bowl_team    238 non-null    object        
 4   inns         238 non-null    int64         
 5   batter_id    238 non-null    int64         
 6   batter       238 non-null    object        
 7   runs         238 non-null    int64         
 8   balls        238 non-null    int64         
 9   isOut        238 non-null    int32         
 10  wicketType   238 non-null    object        
 11  venue        238 non-null    object        
 12  boundaries   238 non-null    int64         
 13  career_avg   237 non-null    float64       
 14  career_rpm   237 non-null    float64       
 15  career_sr    237 non-null    float64       
 16  ca

# Venue Stats

In [11]:
# Group by batter and venue to compute cumulative statistics
grouped = df.groupby(['batter_id', 'venue'])

# Compute cumulative statistics
df['cumulative_runs'] = grouped['runs'].cumsum() - df['runs']
df['cumulative_dismissals'] = grouped['isOut'].cumsum() - df['isOut']
df['cumulative_boundaries'] = grouped['boundaries'].cumsum() - df['boundaries']
df['cumulative_balls'] = grouped['balls'].cumsum() - df['balls']
df['cumulative_matches'] = grouped.cumcount()

# Compute batting statistics
df['ven_avg'] = df['cumulative_runs'] / df['cumulative_dismissals'].replace(0, 1)
df['ven_rpm'] = df['cumulative_runs'] / df['cumulative_matches']
df['ven_sr'] = (df['cumulative_runs'] * 100) / df['cumulative_balls'].replace(0, 1)
df['ven_bpm'] = df['cumulative_boundaries'] / df['cumulative_matches']
df['ven_bfpm'] = df['cumulative_balls'] / df['cumulative_matches']

# Drop temporary columns
df.drop(columns=['cumulative_runs',	'cumulative_dismissals',	'cumulative_balls',	'cumulative_matches',	'cumulative_boundaries'], inplace=True)

# Columns to process
venue_cols = ['ven_avg', 'ven_bfpm', 'ven_sr', 'ven_rpm', 'ven_bpm']

# making the first val as Null
df.loc[df.groupby(['batter', 'venue']).head(1).index, venue_cols] = pd.NA

In [12]:
df[(df['batter_id'] == 234675) & (df['venue'] == 'MA Chidambaram Stadium')].head()

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,isOut,...,inns_avg,inns_rpm,inns_sr,inns_bpm,inns_bfpm,ven_avg,ven_rpm,ven_sr,ven_bpm,ven_bfpm
19497,2011-05-18,501261,Kochi Tuskers Kerala,Chennai Super Kings,2,234675,Ravindra Jadeja,19,13,1,...,28.909091,13.826087,110.801394,1.347826,12.478261,NaN,NaN,NaN,NaN,NaN
19720,2012-04-04,548306,Chennai Super Kings,Mumbai Indians,1,234675,Ravindra Jadeja,3,5,1,...,25.066667,22.117647,126.599327,2.647059,17.470588,19.0,19.000000,146.153846,2.0,13.000000
19994,2012-04-12,548318,Chennai Super Kings,Royal Challengers Bangalore,2,234675,Ravindra Jadeja,4,1,0,...,28.083333,14.041667,112.333333,1.375000,12.500000,11.0,11.000000,122.222222,1.0,9.000000
20246,2012-04-19,548330,Chennai Super Kings,Pune Warriors,1,234675,Ravindra Jadeja,7,3,1,...,25.473684,23.047619,129.411765,2.761905,17.809524,13.0,8.666667,136.842105,1.0,6.333333
20304,2012-04-21,548332,Chennai Super Kings,Rajasthan Royals,2,234675,Ravindra Jadeja,0,0,0,...,28.416667,13.640000,113.289037,1.360000,12.040000,11.0,8.250000,150.000000,1.0,5.500000


In [13]:
print("No of unique venues played at by Ravindra Jadeja : ",len(df[(df['batter_id'] == 234675)]['venue'].unique()))
df[(df['batter_id'] == 234675)].info() # NOTE - Venue stats can have as many row of null values for each batter as the number of venues played at

No of unique venues played at by Ravindra Jadeja :  31
<class 'pandas.core.frame.DataFrame'>
Int64Index: 238 entries, 14317 to 14143
Data columns (total 28 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         238 non-null    datetime64[ns]
 1   match_id     238 non-null    int64         
 2   bat_team     238 non-null    object        
 3   bowl_team    238 non-null    object        
 4   inns         238 non-null    int64         
 5   batter_id    238 non-null    int64         
 6   batter       238 non-null    object        
 7   runs         238 non-null    int64         
 8   balls        238 non-null    int64         
 9   isOut        238 non-null    int32         
 10  wicketType   238 non-null    object        
 11  venue        238 non-null    object        
 12  boundaries   238 non-null    int64         
 13  career_avg   237 non-null    float64       
 14  career_rpm   237 non-null    float64       
 

# Season Stats

In [14]:
df['season'] = df['date'].dt.year

# Group by batter and venue to compute cumulative statistics
grouped = df.groupby(['batter_id', 'season'])

# Compute cumulative statistics
df['cumulative_runs'] = grouped['runs'].cumsum() - df['runs']
df['cumulative_dismissals'] = grouped['isOut'].cumsum() - df['isOut']
df['cumulative_boundaries'] = grouped['boundaries'].cumsum() - df['boundaries']
df['cumulative_balls'] = grouped['balls'].cumsum() - df['balls']
df['cumulative_matches'] = grouped.cumcount()

# Compute batting statistics
df['ssn_avg'] = df['cumulative_runs'] / df['cumulative_dismissals'].replace(0, 1)
df['ssn_rpm'] = df['cumulative_runs'] / df['cumulative_matches']
df['ssn_sr'] = (df['cumulative_runs'] * 100) / df['cumulative_balls'].replace(0, 1)
df['ssn_bpm'] = df['cumulative_boundaries'] / df['cumulative_matches']
df['ssn_bfpm'] = df['cumulative_balls'] / df['cumulative_matches']

# Drop temporary columns
df.drop(columns=['cumulative_runs',	'cumulative_dismissals',	'cumulative_balls',	'cumulative_matches',	'cumulative_boundaries'], inplace=True)

# Columns to process
ssn_cols = ['ssn_avg', 'ssn_bfpm', 'ssn_sr', 'ssn_rpm', 'ssn_bpm']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in ssn_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['batter', 'season']).head(1).index, ssn_cols] = pd.NA

In [15]:
df[(df['batter_id'] == 234675) & (df['season'] == 2015)].head()

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,isOut,...,ven_rpm,ven_sr,ven_bpm,ven_bfpm,season,ssn_avg,ssn_rpm,ssn_sr,ssn_bpm,ssn_bfpm
25,2015-04-09,829707,Chennai Super Kings,Delhi Capitals,1,234675,Ravindra Jadeja,17,18,1,...,9.210526,130.597015,0.894737,7.052632,2015,NaN,NaN,NaN,NaN,NaN
70,2015-04-11,829711,Chennai Super Kings,Sunrisers Hyderabad,1,234675,Ravindra Jadeja,0,1,1,...,9.600000,126.315789,0.950000,7.600000,2015,17.0,17.000000,94.444444,2.000000,18.000000
285,2015-04-17,829729,Chennai Super Kings,Mumbai Indians,2,234675,Ravindra Jadeja,0,0,0,...,15.666667,142.424242,2.000000,11.000000,2015,8.5,8.500000,89.473684,1.000000,9.500000
340,2015-04-19,829735,Chennai Super Kings,Rajasthan Royals,1,234675,Ravindra Jadeja,0,0,0,...,NaN,NaN,NaN,NaN,2015,8.5,5.666667,89.473684,0.666667,6.333333
447,2015-04-22,829745,Chennai Super Kings,Royal Challengers Bangalore,1,234675,Ravindra Jadeja,8,8,1,...,6.200000,110.714286,0.600000,5.600000,2015,8.5,4.250000,89.473684,0.500000,4.750000


In [16]:
print("No of unique seasons played by Ravindra Jadeja : ",len(df[(df['batter_id'] == 234675)]['season'].unique()))
df[(df['batter_id'] == 234675)].info() # NOTE - Season stats can have as many row of null values for each batter as the number of seasons played

No of unique seasons played by Ravindra Jadeja :  16
<class 'pandas.core.frame.DataFrame'>
Int64Index: 238 entries, 14317 to 14143
Data columns (total 34 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         238 non-null    datetime64[ns]
 1   match_id     238 non-null    int64         
 2   bat_team     238 non-null    object        
 3   bowl_team    238 non-null    object        
 4   inns         238 non-null    int64         
 5   batter_id    238 non-null    int64         
 6   batter       238 non-null    object        
 7   runs         238 non-null    int64         
 8   balls        238 non-null    int64         
 9   isOut        238 non-null    int32         
 10  wicketType   238 non-null    object        
 11  venue        238 non-null    object        
 12  boundaries   238 non-null    int64         
 13  career_avg   237 non-null    float64       
 14  career_rpm   237 non-null    float64       
 15

# Last5 Match Stats

In [17]:
# Sort batting data by batter and match date
df.sort_values(by=['batter', 'date'], inplace=True)

# ------------------------------------------------------------------------------
# 1) batter Last 5 Match Rolling batter's Average
# ------------------------------------------------------------------------------
rolling_runs = df.groupby('batter_id')['runs'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
rolling_dismissals = df.groupby('batter_id')['isOut'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
).replace(0,1)

df['last5_avg'] = rolling_runs / rolling_dismissals

# ------------------------------------------------------------------------------
# 2) batter Last 5 Match Rolling Strike Rate
# ------------------------------------------------------------------------------
rolling_runs = df.groupby('batter_id')['runs'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
rolling_balls = df.groupby('batter_id')['balls'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
).replace(0,1
          )
df['last5_sr'] = (rolling_runs)*100 / rolling_balls

# ------------------------------------------------------------------------------
# 3) batter Last 5 Match Rolling runs per match
# ------------------------------------------------------------------------------
df['last5_rpm'] = df.groupby('batter_id')['runs'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
)

# ------------------------------------------------------------------------------
# 4) batter Last 5 Match Rolling ball faced per Match
# ------------------------------------------------------------------------------
df['last5_bfpm'] = df.groupby('batter_id')['balls'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
)

# ------------------------------------------------------------------------------
# 5) batter Last 5 Match Rolling boundaries faced per Match
# ------------------------------------------------------------------------------
df['last5_bpm'] = df.groupby('batter_id')['boundaries'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
)

In [18]:
df[(df['batter_id'] == 234675)].head()

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,isOut,...,ssn_avg,ssn_rpm,ssn_sr,ssn_bpm,ssn_bfpm,last5_avg,last5_sr,last5_rpm,last5_bfpm,last5_bpm
14317,2008-04-19,335984,Rajasthan Royals,Delhi Capitals,1,234675,Ravindra Jadeja,29,23,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14394,2008-04-21,335987,Rajasthan Royals,Punjab Kings,2,234675,Ravindra Jadeja,36,25,0,...,29.0,29.0,126.086957,4.000000,23.00,29.0,126.086957,29.0,23.00,4.000000
14460,2008-04-24,335990,Rajasthan Royals,Deccan Chargers,2,234675,Ravindra Jadeja,1,3,1,...,65.0,32.5,135.416667,5.000000,24.00,65.0,135.416667,32.5,24.00,5.000000
14506,2008-04-26,335992,Rajasthan Royals,Royal Challengers Bangalore,2,234675,Ravindra Jadeja,0,0,0,...,33.0,22.0,129.411765,3.333333,17.00,33.0,129.411765,22.0,17.00,3.333333
14669,2008-05-01,336000,Rajasthan Royals,Kolkata Knight Riders,1,234675,Ravindra Jadeja,33,19,1,...,33.0,16.5,129.411765,2.500000,12.75,33.0,129.411765,16.5,12.75,2.500000


In [19]:
df[(df['batter_id'] == 234675)].info() # NOTE - Last5 stats can have one row of null values for each batter

<class 'pandas.core.frame.DataFrame'>
Int64Index: 238 entries, 14317 to 14143
Data columns (total 39 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         238 non-null    datetime64[ns]
 1   match_id     238 non-null    int64         
 2   bat_team     238 non-null    object        
 3   bowl_team    238 non-null    object        
 4   inns         238 non-null    int64         
 5   batter_id    238 non-null    int64         
 6   batter       238 non-null    object        
 7   runs         238 non-null    int64         
 8   balls        238 non-null    int64         
 9   isOut        238 non-null    int32         
 10  wicketType   238 non-null    object        
 11  venue        238 non-null    object        
 12  boundaries   238 non-null    int64         
 13  career_avg   237 non-null    float64       
 14  career_rpm   237 non-null    float64       
 15  career_sr    237 non-null    float64       
 16  ca

# Cleaning and Exporting

In [20]:
df.drop(columns = ['runs', 'balls', 'isOut', 'wicketType', 'boundaries'], inplace=True)

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24333 entries, 14699 to 3812
Data columns (total 34 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         24333 non-null  datetime64[ns]
 1   match_id     24333 non-null  int64         
 2   bat_team     24333 non-null  object        
 3   bowl_team    24333 non-null  object        
 4   inns         24333 non-null  int64         
 5   batter_id    24333 non-null  int64         
 6   batter       24333 non-null  object        
 7   venue        24333 non-null  object        
 8   career_avg   23600 non-null  float64       
 9   career_rpm   23600 non-null  float64       
 10  career_sr    23600 non-null  float64       
 11  career_bpm   23600 non-null  float64       
 12  career_bfpm  23600 non-null  float64       
 13  inns_avg     22961 non-null  float64       
 14  inns_rpm     22961 non-null  float64       
 15  inns_sr      22961 non-null  float64       
 16  i

In [22]:
df['date'].min(), df['date'].max()

(Timestamp('2008-04-18 00:00:00'), Timestamp('2024-05-26 00:00:00'))

In [23]:
df.to_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\processed_data\ipl_bat_features.csv", index=False)